# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
from MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake


**Mixed info**\
'ADNIMERGE', \
'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

**Single Cofactor**\
'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

**Volumes**\
'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',\
'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

**CSF**\
'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [2]:
file_codes = ['UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS']
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

7


In [3]:
dfs = {}
df_names = {}
df_code = []
for idx, (file_name, df_raw) in enumerate(zip_files.items()):
    df_copy = df_raw.copy(deep=True)
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name)

time_buffer = pd.Timedelta(days=80)

0 ---> UCSFFSX_11_02_15_11Aug2025_04.csv
1 ---> UCSFFSX7_11Aug2025_04.csv
2 ---> UCSFFSX6_11Aug2025_04.csv
3 ---> UCSFFSX51_11_08_19_11Aug2025_04.csv
4 ---> UCSFFSL_02_01_16_11Aug2025_04.csv
5 ---> UPENNROI_MARS_06_01_16_09Oct2025_04.csv
6 ---> UCSDVOL_28Oct2025_04.csv


# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [4]:

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'VISCODE'])
print("righe con stessi ####### RID-VISCODE:")
display(subj_viscode_matrix)

righe con stessi ####### RID:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,844,,,,,,
df_1,11,807,,,,,
df_2,64,281,1122,,,,
df_3,82,73,318,1067,,,
df_4,755,11,63,82,755,,
df_5,840,11,64,82,753,840,
df_6,735,11,64,78,679,734,736


righe con stessi ####### RID-EXAMDATE: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4143,,,,,,
df_1,0,847,,,,,
df_2,0,0,2222,,,,
df_3,5,0,0,4350,,,
df_4,3503,0,0,3,3504,,
df_5,838,0,0,0,745,840,
df_6,2569,0,0,2,2474,732,2597


righe con stessi ####### RID-VISCODE:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4087,,,,,,
df_1,0,845,,,,,
df_2,0,0,2220,,,,
df_3,5,0,0,4346,,,
df_4,3493,0,0,3,3501,,
df_5,838,0,0,0,744,840,
df_6,2567,0,0,2,2470,734,2597


# Inizio Merge
## Definizione di df_base e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_base che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [5]:
df_base = dfs['df_0'].copy(deep=True)                           #sembra un errore ma questi df sono definiti
idx_add = [ 'df_3', 'df_4', 'df_6', 'df_5','df_2', 'df_1',]
merge_contains = ['df_0']
i = 0

## Approfondimento RID-EXAMDATE
1. vedo quante righe ci sono con match ESATTO e quante con TIME BUFFER.

In [30]:
df_add = dfs[idx_add[i]].copy(deep=True)
print(idx_add[i])

df_4


In [31]:
buff_index1, buff_index2 = mergeTools.compare_matches_with_without_buffer(df_base, df_add, time_buffer=time_buffer, print_info=True)
all_index1, all_index2 = mergeTools.get_date_match_index(df_base, df_add, time_buffer=time_buffer)

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match con buffer SPAIATI')

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match SPAIATI')

[date_matches_with_buffer] Trovati solo match esatti --> 3503.
ZERO matches con TIME BUFFER


In [32]:
columns_in_common = list(df_base.columns.intersection(df_add.columns))
columns_only_base = list(df_base.columns.difference(df_add.columns))
columns_only_add = list(df_add.columns.difference(df_base.columns))

# studio le colonne in comune e non ai due df
print('Le colonne in comune sono:\n', columns_in_common)
print('\n\nLe colonne solo in df_base sono: \n', columns_only_base)
print('\n\nLe colonne solo in df_add sono: \n', columns_only_add)
print('__________________________________________________________________\n\n')

diff_date = False
# verifico ci siano righe matchate con BUFFER
if len(buff_index1) == len(buff_index2) and len(buff_index1) > 0:    
    diff_date = True
    # verifico se le righe matchate sono TUTTE matchate con BUFFER
    if all(all_index1) == len(all_index2) and all_index1 == buff_index1 and all_index2 == buff_index2:
        print(f'all maches have a buffer, tot: {len(all_index1)} matches\n\n====================================> Renamen\'EXAMDATE\' column\n')
    else:
        print(f'There are {len(buff_index1)} match with buffer\nThere are {len(all_index1)-len(buff_index1)} match exact\nOver {len(all_index1)} total matches\n\n====================================> SHOULD \'EXAMDATE\' column be renamed?\n')
    

# ci sono solo match ESATTI
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) > 0:
    print('JUST exact matches')
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) == 0:
    print('NO matches')


Le colonne in comune sono:
 ['RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'STATUS', 'ICV%ICV', 'MidTemp%ICV', 'Fusiform%ICV', 'Ventricles%ICV', 'Entorhinal%ICV', 'Hippocampus%ICV']


Le colonne solo in df_base sono: 
 ['COHORT']


Le colonne solo in df_add sono: 
 []
__________________________________________________________________


JUST exact matches


In [33]:
if diff_date:
    col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
    temp_merge = mergeTools.create_temp_merge(df_base, df_add, all_index1, all_index2, col_list=col_list)
    display(temp_merge)
    diff = temp_merge['EXAMDATE_1']-temp_merge['EXAMDATE_2']
    display(diff)

In [34]:
modify_examdate = False #True   #False
if modify_examdate:
    df_add.loc[all_index2, 'EXAMDATE'] = df_base.loc[all_index1, 'EXAMDATE'].values
    display(df_add.loc[all_index2]['EXAMDATE'])

In [35]:
rename_examdate = False
rename_df_base = False
new_name = 'MRI_SCANDATE'                 # 'MRI_SCANDATE', 'PET_SCANDATE', 'CSF_DATE', 'PLASMA_DATE'

if rename_examdate:
    if rename_df_base:
        df_base.rename(columns={'EXAMDATE': new_name}, inplace=True)
    else:
        df_add.rename(columns={'EXAMDATE': new_name}, inplace=True)
else:
    print('EXAMDATE column NOT renamed')


EXAMDATE column NOT renamed


## Studio Colonne in comune per righe che matchano
1) Se ci sono righe che machano tra i due df allora identifico altre colonne in comune ai due df.

2) Faccio merge tra i due df escludendo i soggetti che hanno visite metchate tra i 2 df (righe). --> merge_base solo aggiunta di soggetti nuovi.

3) Quindi se ci sono colonne in comune e righe che matchano faccio merge soggetto per soggetto (tra i soggetti con  visite in entrambi i df).\
Aggiungo qusti merge di singoli soggetti al resto del merge_base.


Così ottengo Merge finale.

In [36]:
reference_col = ['RID', 'VISCODE', 'EXAMDATE', 'VISIT_MONTH', 'COHORT'] #più altre specifiche

if len(all_index1) > 0 and len(columns_in_common) > 0:
    print('TO DO: merge specifico per soggetti con visite comuni ai 2 df')
    print('---> Colonne comuni specifiche:\n', [x for x in columns_in_common if x not in reference_col])
elif len(columns_in_common) == 0:
    print('merge NON possibile, non ci sono colonne comuni tra i 2 df')
elif len(all_index1) == 0:
    print('EASY! Puoi mergiare direttamente, NON si sovrappongono RIGHE')


TO DO: merge specifico per soggetti con visite comuni ai 2 df
---> Colonne comuni specifiche:
 ['STATUS', 'ICV%ICV', 'MidTemp%ICV', 'Fusiform%ICV', 'Ventricles%ICV', 'Entorhinal%ICV', 'Hippocampus%ICV']


In [37]:
rid_list = df_base.iloc[all_index1]['RID'].unique().tolist()
print(len(rid_list))

if len(rid_list) > 0:
    df_a = df_base[~df_base['RID'].isin(rid_list)].copy(deep=True)
    df_b = df_add[~df_add['RID'].isin(rid_list)].copy(deep=True)
    # merge per i soggetti che non hanno visite comuni
    df_merge = pd.merge(df_a, df_b, how='outer')
    print('df_a + df_b = ', len(df_a)+ len(df_b), '\n df_merge = ', len(df_merge))

    # merge soggetto per soggetto con attenzione per colonne comuni e valori che si sovrappongono 
    ref_col = [x for x in reference_col if x in columns_in_common and x != 'RID']
    df_merge_sub = None
    for rid in rid_list:
        print(rid)
        single_sub_df = mergeTools.merge_paired_rows_rid_specific(df_base, df_add, all_index1, all_index2, ref_col, subject_id=rid)
        if df_merge_sub is None:
            df_merge_sub = single_sub_df.copy(deep=True)
        else:
            df_merge_sub = pd.merge(df_merge_sub, single_sub_df, how='outer')
        #print(rid, len(merged_sub_df), len(df_merge))
    print('\nmerge soggetti', len(df_merge_sub))

else:
    df_merge = pd.merge(df_base, df_add, how='outer')
    print('df_base + df_add = ', len(df_base)+ len(df_add), '\n df_merge = ', len(df_merge))

755
df_a + df_b =  4276 
 df_merge =  4276
3
3 
### There are differences between EXAMDATE_1 and EXAMDATE_2 ---> should be handled before
### ATTENZIONE: MidTemp%ICV diff >>> 10% media
### ATTENZIONE: Fusiform%ICV diff >>> 10% media
### ATTENZIONE: Ventricles%ICV diff >>> 10% media
### ATTENZIONE: Entorhinal%ICV diff >>> 10% media
4
4 
### There are differences between EXAMDATE_1 and EXAMDATE_2 ---> should be handled before
### ATTENZIONE: Ventricles%ICV diff >>> 10% media
### ATTENZIONE: Entorhinal%ICV diff >>> 10% media
### ATTENZIONE: Hippocampus%ICV diff >>> 10% media
5
5 
### There are differences between EXAMDATE_1 and EXAMDATE_2 ---> should be handled before
### ATTENZIONE: Ventricles%ICV diff >>> 10% media
### ATTENZIONE: Entorhinal%ICV diff >>> 10% media
### ATTENZIONE: Hippocampus%ICV diff >>> 10% media
6
6 
### There are differences between EXAMDATE_1 and EXAMDATE_2 ---> should be handled before
### ATTENZIONE: Ventricles%ICV diff >>> 10% media
### ATTENZIONE: Hippocampus%IC

In [27]:
if len(rid_list) > 0:
    df_merge = pd.merge(df_merge, df_merge_sub, how='outer')
    print('merge finale', len(df_merge))

df_merge = df_merge.sort_values(by=['RID', 'EXAMDATE'], ascending=[True, True])

merge finale 8488


In [28]:
import json
if i == 0:
    prev = 0
# Apri e carica il file diff_tracking.json
if os.path.exists('diff_tracking.json'):
    with open('diff_tracking.json', 'r') as f:
        diff_tracking = json.load(f)
    
    # Ad esempio, mostriamo le chiavi e il primo valore
    print("Chiavi disponibili in diff_tracking.json:", list(diff_tracking.keys()))
    for key in diff_tracking.keys():
        print(key, len(diff_tracking[key]),'/',len(rid_list) + prev)
        prev += len(rid_list)
else:
    print("JSON non creato: il file diff_tracking.json non esiste")


Chiavi disponibili in diff_tracking.json: ['MidTemp%ICV', 'Entorhinal%ICV', 'Fusiform%ICV']
MidTemp%ICV 1 / 5
Entorhinal%ICV 2 / 10
Fusiform%ICV 1 / 15


In [29]:
df_base = df_merge.copy(deep=True)
merge_contains.append(idx_add[i])
#prepare for next merge
i += 1
print(f'il merg contine i seguenti df: {merge_contains}')
print(f'il prossimo df da unire è: {idx_add[i]}\n\n ===> torna al capitolo: "Approfondimento RID-EXAMDATE"')


il merg contine i seguenti df: ['df_0', 'df_3']
il prossimo df da unire è: df_4

 ===> torna al capitolo: "Approfondimento RID-EXAMDATE"


In [ ]:
if os.path.exists("diff_tracking.json"):
    os.remove("diff_tracking.json")

# Verifiche specifiche

In [38]:
col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
temp_merge = mergeTools.create_temp_merge(df_base, df_add, all_index1, all_index2, rid=1225 , col_list=col_list)
merged_sub_df = mergeTools.merge_paired_rows_rid_specific(df_base, df_add, all_index1, all_index2, ref_col, subject_id=1225 )

    

1225 
### There are differences between EXAMDATE_1 and EXAMDATE_2 ---> should be handled before
### ATTENZIONE: MidTemp%ICV diff >>> 10% media
### ATTENZIONE: Fusiform%ICV diff >>> 10% media
### ATTENZIONE: Ventricles%ICV diff >>> 10% media
### ATTENZIONE: Entorhinal%ICV diff >>> 10% media
### ATTENZIONE: Hippocampus%ICV diff >>> 10% media


In [39]:
temp_merge

,RID,VISCODE_1,VISCODE_2,VISIT_MONTH_1,VISIT_MONTH_2,EXAMDATE_1,EXAMDATE_2,STATUS_1,STATUS_2,ICV%ICV_1,...,Fusiform%ICV_1,Fusiform%ICV_2,Ventricles%ICV_1,Ventricles%ICV_2,Entorhinal%ICV_1,Entorhinal%ICV_2,Hippocampus%ICV_1,Hippocampus%ICV_2,COHORT_1,COHORT_2
0,1225,m06,sc,7.0,0,2007-08-20,2007-01-24,complete,complete,100.0,...,1.021308,1.069428,1.939311,1.897212,0.190183,0.158929,0.251316,0.260188,NaN,None
1,1225,m12,m06,13.0,7,2008-02-11,2007-08-20,complete,complete,100.0,...,0.899614,1.036337,2.000518,1.972097,0.161196,0.157334,0.249275,0.234638,NaN,None
2,1225,m18,m12,19.0,13,2008-09-08,2008-02-11,complete,complete,100.0,...,0.951319,0.980559,2.083774,2.011630,0.113512,0.155260,0.227277,0.227429,NaN,None
3,1225,sc,m18,0.0,19,2007-01-24,2008-09-08,complete,complete,100.0,...,0.959694,1.001198,1.880006,2.120661,0.144400,0.146089,0.276119,0.223348,NaN,None


In [ ]:
merged_sub_df

# altro